<a href="https://www.kaggle.com/code/mrrogueknight/vandermonde-polynomial-solver-tarpeen-data?scriptVersionId=335798293" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [34]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [35]:
"""
Prediction Calculator

A professional scientific computing tool for estimating Y values from X
with multiple interpolation methods, automatic model selection,
and comprehensive diagnostics.

Features:
    - Multiple interpolation methods (Polynomial, Spline, PCHIP, Linear)
    - Automatic model selection (LOOCV, AIC, BIC)
    - Method dispatcher for reproducible results
    - LOOCV-based uncertainty estimation
    - Model agreement intervals
    - Transparent reliability scoring
    - Professional interface

Author: Scientific Computing Framework
License: MIT
"""

from __future__ import annotations

import numpy as np
from dataclasses import dataclass
from enum import Enum
from typing import Optional, Tuple, List, Dict, Any, Callable
import logging
from math import comb
import time
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

# ===================================================================
# MATPLOTLIB STYLE
# ===================================================================

try:
    plt.style.use('seaborn-v0-8-darkgrid')
except OSError:
    try:
        plt.style.use('seaborn-darkgrid')
    except OSError:
        try:
            plt.style.use('dark_background')
        except OSError:
            pass

mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'Liberation Serif']
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.labelsize'] = 11
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['legend.fontsize'] = 9
mpl.rcParams['figure.titlesize'] = 13
mpl.rcParams['figure.dpi'] = 100

# ===================================================================
# CONSTANTS
# ===================================================================

MACHINE_EPSILON = np.finfo(np.float64).eps
DEFAULT_DEGREE = 6
PLOT_POINTS = 500

# Default data
DEFAULT_X = np.array([30.75, 30.88, 31.00, 31.12, 31.25, 31.38, 31.50, 31.62, 31.75, 31.88])
DEFAULT_Y = np.array([1056.6621, 1062.4049, 1059.5334, 1061.4478, 1060.0, 1061.0, 1060.5, 1061.5, 1060.8, 1061.2])

# ===================================================================
# ENUMS
# ===================================================================

class PredictionMode(Enum):
    """Prediction mode enumeration."""
    AUTOMATIC = "automatic"
    LINEAR = "linear"
    POLYNOMIAL = "polynomial"
    SPLINE = "spline"
    PCHIP = "pchip"
    CUBIC = "cubic"

class ReliabilityLevel(Enum):
    """Reliability levels for predictions."""
    EXCELLENT = "Excellent"
    GOOD = "Good"
    MODERATE = "Moderate"
    LOW = "Low"
    VERY_LOW = "Very Low"

# ===================================================================
# DATA CLASSES
# ===================================================================

@dataclass(slots=True)
class ModelResult:
    """Model result with diagnostics."""
    method: str
    degree: int
    coefficients: np.ndarray
    expanded_coeffs: np.ndarray
    shift: float
    scale: float
    r_squared: float
    adjusted_r_squared: float
    rmse: float
    loocv_error: float
    aic: float
    bic: float
    condition_number: float
    residual_norm: float
    prediction: Optional[float] = None
    prediction_std: Optional[float] = None
    agreement_interval: Optional[Tuple[float, float]] = None

@dataclass(slots=True)
class ReliabilityBreakdown:
    """Reliability breakdown."""
    score: float
    level: str
    components: Dict[str, float]
    reasons: List[str]
    warnings: List[str]

@dataclass(slots=True)
class PredictionResult:
    """Prediction result with reliability."""
    x_target: float
    mode: str
    prediction: float
    prediction_std: float
    all_predictions: Dict[str, float]
    prediction_range: Tuple[float, float]
    agreement_interval: Tuple[float, float]
    is_extrapolation: bool
    extrapolation_distance: float
    warning: Optional[str]
    execution_time: float
    loocv_error: float
    reliability: ReliabilityBreakdown
    method_used: str

# ===================================================================
# INTERPOLATION METHODS
# ===================================================================

class PolynomialInterpolator:
    """Polynomial interpolation with shifted Vandermonde basis."""
    
    def __init__(self):
        self._coefficients = None
        self._expanded_coeffs = None
        self._shift = 0.0
        self._scale = 1.0
        self._degree = 0
        self._condition = 1.0
        self._fitted = False
        self._x_data = None
        self._y_data = None
    
    def fit(self, x_data: np.ndarray, y_data: np.ndarray, degree: int) -> None:
        """Fit polynomial of specified degree."""
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        self._x_data = x_data
        self._y_data = y_data
        self._degree = degree
        
        # Shift and scale for numerical stability
        x_min = np.min(x_data)
        x_max = np.max(x_data)
        self._shift = (x_min + x_max) / 2.0
        half_range = (x_max - x_min) / 2.0
        self._scale = 1.0 / (half_range + MACHINE_EPSILON)
        
        shifted_x = (x_data - self._shift) * self._scale
        
        # Build Vandermonde in shifted basis
        V = np.vander(shifted_x, N=degree + 1, increasing=True)
        
        if len(x_data) > degree + 1:
            coeffs, residuals, rank, s = np.linalg.lstsq(V, y_data, rcond=None)
            self._condition = s[0] / (s[-1] + MACHINE_EPSILON) if len(s) > 0 else 1.0
        else:
            coeffs = np.linalg.solve(V, y_data)
            self._condition = np.linalg.cond(V)
        
        self._coefficients = coeffs.flatten()
        self._expanded_coeffs = self._expand_coefficients()
        self._fitted = True
    
    def _expand_coefficients(self) -> np.ndarray:
        """Expand shifted coefficients to original basis."""
        degree = len(self._coefficients) - 1
        expanded = np.zeros(degree + 1, dtype=np.float64)
        
        for i, c in enumerate(self._coefficients):
            if abs(c) < MACHINE_EPSILON:
                continue
            scaled_c = c * (self._scale ** i)
            for j in range(i + 1):
                binom = comb(i, j)
                term = scaled_c * binom * ((-self._shift) ** (i - j))
                expanded[degree - j] += term
        
        return expanded
    
    def predict(self, x_target: float) -> float:
        """Predict using Horner's method."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        shifted_x = (x_target - self._shift) * self._scale
        y = 0.0
        for c in reversed(self._coefficients):
            y = y * shifted_x + c
        return y
    
    def evaluate(self, x_values: np.ndarray) -> np.ndarray:
        """Evaluate at multiple points."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        x_array = np.asarray(x_values, dtype=np.float64).flatten()
        shifted_x = (x_array - self._shift) * self._scale
        y = np.zeros_like(shifted_x)
        for c in reversed(self._coefficients):
            y = y * shifted_x + c
        return y
    
    def loocv_predict(self, x_target: float) -> Tuple[float, float]:
        """Leave-one-out cross-validation prediction."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        n = len(self._x_data)
        predictions = []
        
        for i in range(n):
            x_train = np.delete(self._x_data, i)
            y_train = np.delete(self._y_data, i)
            
            temp = PolynomialInterpolator()
            try:
                temp.fit(x_train, y_train, self._degree)
                pred = temp.predict(x_target)
                predictions.append(pred)
            except:
                continue
        
        if len(predictions) < 2:
            return self.predict(x_target), 0.0
        
        pred_array = np.array(predictions)
        return np.mean(pred_array), np.std(pred_array)
    
    @property
    def coefficients(self):
        return self._coefficients
    
    @property
    def expanded_coefficients(self):
        return self._expanded_coeffs
    
    @property
    def degree(self):
        return self._degree
    
    @property
    def condition_number(self):
        return self._condition
    
    @property
    def shift(self):
        return self._shift
    
    @property
    def scale(self):
        return self._scale
    
    @property
    def x_data(self):
        return self._x_data
    
    @property
    def y_data(self):
        return self._y_data


class LinearInterpolator:
    """Linear interpolation with optional adjacent/non-adjacent selection."""
    
    def __init__(self):
        self._x_data = None
        self._y_data = None
        self._fitted = False
    
    def fit(self, x_data: np.ndarray, y_data: np.ndarray) -> None:
        """Fit linear model."""
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        # Sort
        idx = np.argsort(x_data)
        self._x_data = x_data[idx]
        self._y_data = y_data[idx]
        self._fitted = True
    
    def predict(self, x_target: float, method: str = 'adjacent') -> float:
        """
        Predict using linear interpolation.
        
        method: 'adjacent' (default) or 'global'
        """
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        n = len(self._x_data)
        
        if method == 'global':
            # Least squares line through all points
            coeffs = np.polyfit(self._x_data, self._y_data, 1)
            return np.polyval(coeffs, x_target)
        
        # Adjacent interpolation
        if x_target <= self._x_data[0]:
            # Extrapolate backward using first two points
            x1, y1 = self._x_data[0], self._y_data[0]
            x2, y2 = self._x_data[1], self._y_data[1]
            return self._linear_interp(x_target, x1, y1, x2, y2)
        elif x_target >= self._x_data[-1]:
            # Extrapolate forward using last two points
            x1, y1 = self._x_data[-2], self._y_data[-2]
            x2, y2 = self._x_data[-1], self._y_data[-1]
            return self._linear_interp(x_target, x1, y1, x2, y2)
        else:
            # Find adjacent interval
            for i in range(n - 1):
                if self._x_data[i] <= x_target <= self._x_data[i + 1]:
                    x1, y1 = self._x_data[i], self._y_data[i]
                    x2, y2 = self._x_data[i + 1], self._y_data[i + 1]
                    return self._linear_interp(x_target, x1, y1, x2, y2)
            
            return self._y_data[-1]
    
    def _linear_interp(self, x: float, x1: float, y1: float, x2: float, y2: float) -> float:
        """Linear interpolation formula."""
        dx = x2 - x1
        if abs(dx) < MACHINE_EPSILON:
            return y1
        return y1 + ((x - x1) / dx) * (y2 - y1)
    
    def evaluate(self, x_values: np.ndarray, method: str = 'adjacent') -> np.ndarray:
        """Evaluate at multiple points."""
        return np.array([self.predict(x, method) for x in x_values])
    
    def loocv_predict(self, x_target: float) -> Tuple[float, float]:
        """LOOCV prediction."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        n = len(self._x_data)
        predictions = []
        
        for i in range(n):
            x_train = np.delete(self._x_data, i)
            y_train = np.delete(self._y_data, i)
            
            temp = LinearInterpolator()
            try:
                temp.fit(x_train, y_train)
                pred = temp.predict(x_target)
                predictions.append(pred)
            except:
                continue
        
        if len(predictions) < 2:
            return self.predict(x_target), 0.0
        
        pred_array = np.array(predictions)
        return np.mean(pred_array), np.std(pred_array)
    
    @property
    def x_data(self):
        return self._x_data
    
    @property
    def y_data(self):
        return self._y_data


class SplineInterpolator:
    """Natural cubic spline interpolation."""
    
    def __init__(self):
        self._fitted = False
        self._x_data = None
        self._y_data = None
    
    def fit(self, x_data: np.ndarray, y_data: np.ndarray) -> None:
        """Fit natural cubic spline."""
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        n = len(x_data)
        if n < 3:
            raise ValueError("Need at least 3 points for spline")
        
        idx = np.argsort(x_data)
        x_data = x_data[idx]
        y_data = y_data[idx]
        
        self._x_data = x_data
        self._y_data = y_data
        
        try:
            from scipy.interpolate import CubicSpline
            self._spline = CubicSpline(x_data, y_data, bc_type='natural')
            self._fitted = True
            return
        except ImportError:
            pass
        
        # Fallback: piecewise polynomials
        self._segments = []
        for i in range(n - 1):
            x_seg = x_data[max(0, i-1):min(n, i+2)]
            y_seg = y_data[max(0, i-1):min(n, i+2)]
            if len(x_seg) >= 2:
                coeffs = np.polyfit(x_seg, y_seg, min(2, len(x_seg)-1))
                self._segments.append({
                    'x_start': x_data[i],
                    'x_end': x_data[i+1],
                    'coeffs': coeffs
                })
        
        self._fitted = True
    
    def predict(self, x_target: float) -> float:
        """Predict at target point."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        if hasattr(self, '_spline'):
            return float(self._spline(x_target))
        
        for seg in self._segments:
            if seg['x_start'] <= x_target <= seg['x_end']:
                return float(np.polyval(seg['coeffs'], x_target))
        
        if x_target < self._x_data[0]:
            return float(np.polyval(self._segments[0]['coeffs'], x_target))
        else:
            return float(np.polyval(self._segments[-1]['coeffs'], x_target))
    
    def evaluate(self, x_values: np.ndarray) -> np.ndarray:
        """Evaluate at multiple points."""
        return np.array([self.predict(x) for x in x_values])
    
    def loocv_predict(self, x_target: float) -> Tuple[float, float]:
        """LOOCV prediction."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        n = len(self._x_data)
        predictions = []
        
        for i in range(n):
            x_train = np.delete(self._x_data, i)
            y_train = np.delete(self._y_data, i)
            
            temp = SplineInterpolator()
            try:
                temp.fit(x_train, y_train)
                pred = temp.predict(x_target)
                predictions.append(pred)
            except:
                continue
        
        if len(predictions) < 2:
            return self.predict(x_target), 0.0
        
        pred_array = np.array(predictions)
        return np.mean(pred_array), np.std(pred_array)
    
    @property
    def x_data(self):
        return self._x_data
    
    @property
    def y_data(self):
        return self._y_data


class PCHIPInterpolator:
    """PCHIP (shape-preserving) interpolation."""
    
    def __init__(self):
        self._fitted = False
        self._x_data = None
        self._y_data = None
    
    def fit(self, x_data: np.ndarray, y_data: np.ndarray) -> None:
        """Fit PCHIP interpolator."""
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        self._x_data = x_data
        self._y_data = y_data
        
        try:
            from scipy.interpolate import PchipInterpolator
            self._pchip = PchipInterpolator(x_data, y_data)
            self._fitted = True
        except ImportError:
            self._poly = PolynomialInterpolator()
            self._poly.fit(x_data, y_data, min(len(x_data)-1, 3))
            self._fitted = True
    
    def predict(self, x_target: float) -> float:
        """Predict at target point."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        if hasattr(self, '_pchip'):
            return float(self._pchip(x_target))
        else:
            return self._poly.predict(x_target)
    
    def evaluate(self, x_values: np.ndarray) -> np.ndarray:
        """Evaluate at multiple points."""
        return np.array([self.predict(x) for x in x_values])
    
    def loocv_predict(self, x_target: float) -> Tuple[float, float]:
        """LOOCV prediction."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        n = len(self._x_data)
        predictions = []
        
        for i in range(n):
            x_train = np.delete(self._x_data, i)
            y_train = np.delete(self._y_data, i)
            
            temp = PCHIPInterpolator()
            try:
                temp.fit(x_train, y_train)
                pred = temp.predict(x_target)
                predictions.append(pred)
            except:
                continue
        
        if len(predictions) < 2:
            return self.predict(x_target), 0.0
        
        pred_array = np.array(predictions)
        return np.mean(pred_array), np.std(pred_array)
    
    @property
    def x_data(self):
        return self._x_data
    
    @property
    def y_data(self):
        return self._y_data

# ===================================================================
# RELIABILITY SCORING
# ===================================================================

class ReliabilityScorer:
    """Reliability scoring with normalized metrics."""
    
    @staticmethod
    def compute_reliability(
        x_target: float,
        x_min: float,
        x_max: float,
        x_range: float,
        loocv_error: float,
        y_std: float,
        model_agreement: float,
        condition_number: float,
        degree: int,
        n_points: int,
        r_squared: float
    ) -> ReliabilityBreakdown:
        """Compute reliability score with breakdown."""
        components = {}
        reasons = []
        warnings = []
        
        # 1. Distance from nearest point (20%)
        if x_min <= x_target <= x_max:
            nearest_dist = min(abs(x_target - x_min), abs(x_target - x_max))
            distance_score = np.exp(-2 * nearest_dist / (x_range + MACHINE_EPSILON))
            reasons.append(f"Inside data range (+{distance_score*20:.0f})")
        else:
            if x_target < x_min:
                dist = (x_min - x_target) / (x_range + MACHINE_EPSILON)
            else:
                dist = (x_target - x_max) / (x_range + MACHINE_EPSILON)
            distance_score = np.exp(-0.5 * dist)
            reasons.append(f"Extrapolation distance {dist:.2f}x (+{distance_score*20:.0f})")
            warnings.append("Extrapolation beyond data range")
        
        components['distance'] = distance_score * 0.20
        
        # 2. LOOCV (25%)
        if y_std > MACHINE_EPSILON:
            normalized_loocv = loocv_error / (y_std + MACHINE_EPSILON)
            loocv_score = 1.0 / (1.0 + normalized_loocv)
        else:
            loocv_score = 1.0
        
        components['loocv'] = loocv_score * 0.25
        
        # 3. Model Agreement (15%)
        if model_agreement > 0:
            normalized_agreement = model_agreement / (y_std + MACHINE_EPSILON)
            agreement_score = 1.0 / (1.0 + normalized_agreement)
        else:
            agreement_score = 1.0
        
        components['agreement'] = agreement_score * 0.15
        
        # 4. Condition Number (10%)
        if condition_number < 100:
            cond_score = 1.0
        elif condition_number < 1000:
            cond_score = 0.85
        elif condition_number < 10000:
            cond_score = 0.60
        elif condition_number < 100000:
            cond_score = 0.30
        else:
            cond_score = 0.10
        
        components['conditioning'] = cond_score * 0.10
        
        # 5. Residual (10%)
        residual_score = min(1.0, r_squared)
        components['residual'] = residual_score * 0.10
        
        # 6. Degree Penalty (5%)
        if degree <= 2:
            degree_score = 1.0
        elif degree <= 3:
            degree_score = 0.85
        elif degree <= 4:
            degree_score = 0.70
        else:
            degree_score = 0.50
        
        components['complexity'] = degree_score * 0.05
        
        # 7. Points (5%)
        points_score = min(1.0, n_points / 10)
        components['points'] = points_score * 0.05
        
        # Calculate final score
        total_score = sum(components.values())
        final_score = total_score * 100
        
        # Determine level
        if final_score >= 90:
            level = "Excellent"
        elif final_score >= 75:
            level = "Good"
        elif final_score >= 55:
            level = "Moderate"
        elif final_score >= 35:
            level = "Low"
        else:
            level = "Very Low"
        
        return ReliabilityBreakdown(
            score=final_score,
            level=level,
            components=components,
            reasons=reasons,
            warnings=warnings
        )

# ===================================================================
# MAIN ENGINE
# ===================================================================

class PredictionEngine:
    """Prediction engine with method dispatcher and automatic selection."""
    
    def __init__(self):
        self._x_data = None
        self._y_data = None
        self._fitted = False
        self._models = {}
        self._best_model_name = None
        self._x_min = None
        self._x_max = None
        self._x_range = None
        self._y_std = 0.0
        self._mode = PredictionMode.AUTOMATIC
        self._linear = None
        self._polynomial = None
        self._spline = None
        self._pchip = None
    
    def set_mode(self, mode: PredictionMode) -> None:
        """Set prediction mode."""
        self._mode = mode
    
    def fit(self, x_data: np.ndarray, y_data: np.ndarray) -> Dict[str, Any]:
        """Fit all models and select the best."""
        x_data = np.asarray(x_data, dtype=np.float64).flatten()
        y_data = np.asarray(y_data, dtype=np.float64).flatten()
        
        if len(x_data) < 3:
            raise ValueError("Need at least 3 data points")
        
        if np.any(~np.isfinite(x_data)) or np.any(~np.isfinite(y_data)):
            raise ValueError("NaN or Inf values detected")
        
        if len(np.unique(x_data)) != len(x_data):
            raise ValueError("Duplicate X values are not allowed")
        
        idx = np.argsort(x_data)
        x_data = x_data[idx]
        y_data = y_data[idx]
        
        self._x_data = x_data
        self._y_data = y_data
        self._x_min = np.min(x_data)
        self._x_max = np.max(x_data)
        self._x_range = self._x_max - self._x_min
        self._y_std = np.std(y_data)
        
        models = {}
        n = len(x_data)
        max_degree = min(n - 1, 6)
        
        # Linear model
        try:
            linear = LinearInterpolator()
            linear.fit(x_data, y_data)
            y_pred = linear.evaluate(x_data)
            residuals = y_data - y_pred
            
            rss = np.sum(residuals**2)
            tss = np.sum((y_data - np.mean(y_data))**2)
            r_squared = 1 - rss / (tss + MACHINE_EPSILON)
            rmse = np.sqrt(rss / n)
            
            # LOOCV
            loocv_errors = []
            for i in range(n):
                x_train = np.delete(x_data, i)
                y_train = np.delete(y_data, i)
                temp = LinearInterpolator()
                try:
                    temp.fit(x_train, y_train)
                    pred = temp.predict(x_data[i])
                    loocv_errors.append((pred - y_data[i])**2)
                except:
                    continue
            
            loocv_error = np.sqrt(np.mean(loocv_errors)) if loocv_errors else rmse
            
            aic = n * np.log(rss / n) + 2 * 2
            bic = n * np.log(rss / n) + 2 * np.log(n)
            
            combined_score = 0.5 * (loocv_error / (self._y_std + MACHINE_EPSILON)) + 0.3 * (rmse / (self._y_std + MACHINE_EPSILON))
            
            models['Linear'] = {
                'model': linear,
                'type': 'linear',
                'degree': 1,
                'r_squared': r_squared,
                'adjusted_r_squared': r_squared,
                'rmse': rmse,
                'loocv_error': loocv_error,
                'aic': aic,
                'bic': bic,
                'condition': 1.0,
                'residual_norm': np.linalg.norm(residuals),
                'combined_score': combined_score
            }
            self._linear = linear
        except Exception as e:
            pass
        
        # Polynomial models
        for degree in range(1, max_degree + 1):
            try:
                poly = PolynomialInterpolator()
                poly.fit(x_data, y_data, degree)
                
                y_pred = poly.evaluate(x_data)
                residuals = y_data - y_pred
                
                rss = np.sum(residuals**2)
                tss = np.sum((y_data - np.mean(y_data))**2)
                r_squared = 1 - rss / (tss + MACHINE_EPSILON)
                
                adjusted_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - degree - 1 + MACHINE_EPSILON)
                rmse = np.sqrt(rss / n)
                
                loocv_errors = []
                for i in range(n):
                    x_train = np.delete(x_data, i)
                    y_train = np.delete(y_data, i)
                    temp = PolynomialInterpolator()
                    try:
                        temp.fit(x_train, y_train, degree)
                        pred = temp.predict(x_data[i])
                        loocv_errors.append((pred - y_data[i])**2)
                    except:
                        continue
                
                loocv_error = np.sqrt(np.mean(loocv_errors)) if loocv_errors else rmse
                
                aic = n * np.log(rss / n) + 2 * (degree + 1)
                bic = n * np.log(rss / n) + (degree + 1) * np.log(n)
                
                norm_loocv = loocv_error / (self._y_std + MACHINE_EPSILON)
                norm_rmse = rmse / (self._y_std + MACHINE_EPSILON)
                norm_cond = np.log10(poly.condition_number + 1) / 10
                
                combined_score = 0.5 * norm_loocv + 0.3 * norm_rmse + 0.2 * norm_cond
                
                models[f'Polynomial (deg {degree})'] = {
                    'model': poly,
                    'type': 'polynomial',
                    'degree': degree,
                    'r_squared': r_squared,
                    'adjusted_r_squared': adjusted_r_squared,
                    'rmse': rmse,
                    'loocv_error': loocv_error,
                    'aic': aic,
                    'bic': bic,
                    'condition': poly.condition_number,
                    'residual_norm': np.linalg.norm(residuals),
                    'combined_score': combined_score
                }
                if degree == 1:
                    self._polynomial = poly
            except Exception as e:
                continue
        
        # Spline model
        try:
            spline = SplineInterpolator()
            spline.fit(x_data, y_data)
            y_pred = spline.evaluate(x_data)
            residuals = y_data - y_pred
            
            rss = np.sum(residuals**2)
            tss = np.sum((y_data - np.mean(y_data))**2)
            r_squared = 1 - rss / (tss + MACHINE_EPSILON)
            rmse = np.sqrt(rss / n)
            
            loocv_errors = []
            for i in range(n):
                x_train = np.delete(x_data, i)
                y_train = np.delete(y_data, i)
                temp = SplineInterpolator()
                try:
                    temp.fit(x_train, y_train)
                    pred = temp.predict(x_data[i])
                    loocv_errors.append((pred - y_data[i])**2)
                except:
                    continue
            
            loocv_error = np.sqrt(np.mean(loocv_errors)) if loocv_errors else rmse
            
            edf = min(n, 10)
            aic = n * np.log(rss / n) + 2 * edf
            bic = n * np.log(rss / n) + edf * np.log(n)
            
            norm_loocv = loocv_error / (self._y_std + MACHINE_EPSILON)
            norm_rmse = rmse / (self._y_std + MACHINE_EPSILON)
            combined_score = 0.5 * norm_loocv + 0.5 * norm_rmse
            
            models['Cubic Spline'] = {
                'model': spline,
                'type': 'spline',
                'degree': 'N/A',
                'r_squared': r_squared,
                'adjusted_r_squared': r_squared,
                'rmse': rmse,
                'loocv_error': loocv_error,
                'aic': aic,
                'bic': bic,
                'condition': 1.0,
                'residual_norm': np.linalg.norm(residuals),
                'combined_score': combined_score
            }
            self._spline = spline
        except Exception as e:
            pass
        
        # PCHIP model
        try:
            pchip = PCHIPInterpolator()
            pchip.fit(x_data, y_data)
            y_pred = pchip.evaluate(x_data)
            residuals = y_data - y_pred
            
            rss = np.sum(residuals**2)
            tss = np.sum((y_data - np.mean(y_data))**2)
            r_squared = 1 - rss / (tss + MACHINE_EPSILON)
            rmse = np.sqrt(rss / n)
            
            loocv_errors = []
            for i in range(n):
                x_train = np.delete(x_data, i)
                y_train = np.delete(y_data, i)
                temp = PCHIPInterpolator()
                try:
                    temp.fit(x_train, y_train)
                    pred = temp.predict(x_data[i])
                    loocv_errors.append((pred - y_data[i])**2)
                except:
                    continue
            
            loocv_error = np.sqrt(np.mean(loocv_errors)) if loocv_errors else rmse
            
            edf = min(n, 8)
            aic = n * np.log(rss / n) + 2 * edf
            bic = n * np.log(rss / n) + edf * np.log(n)
            
            norm_loocv = loocv_error / (self._y_std + MACHINE_EPSILON)
            norm_rmse = rmse / (self._y_std + MACHINE_EPSILON)
            combined_score = 0.5 * norm_loocv + 0.5 * norm_rmse
            
            models['PCHIP'] = {
                'model': pchip,
                'type': 'pchip',
                'degree': 'N/A',
                'r_squared': r_squared,
                'adjusted_r_squared': r_squared,
                'rmse': rmse,
                'loocv_error': loocv_error,
                'aic': aic,
                'bic': bic,
                'condition': 1.0,
                'residual_norm': np.linalg.norm(residuals),
                'combined_score': combined_score
            }
            self._pchip = pchip
        except Exception as e:
            pass
        
        if not models:
            raise ValueError("No models could be fitted")
        
        self._models = models
        
        # Select best model using combined score
        best_name = min(models.keys(), key=lambda k: models[k]['combined_score'])
        self._best_model_name = best_name
        
        self._fitted = True
        
        return {
            'models': models,
            'best_model': best_name,
            'n_points': n,
            'best_score': models[best_name]['combined_score']
        }
    
    def predict(self, x_target: float) -> PredictionResult:
        """Predict using selected mode."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        start_time = time.time()
        
        # Check extrapolation
        is_extrap = not (self._x_min <= x_target <= self._x_max)
        if is_extrap:
            if x_target < self._x_min:
                distance = (self._x_min - x_target) / (self._x_range + MACHINE_EPSILON)
            else:
                distance = (x_target - self._x_max) / (self._x_range + MACHINE_EPSILON)
            
            if distance > 2.0:
                warning = f"Value is {distance:.1f}x outside the data range"
            elif distance > 0.5:
                warning = "Value is outside the supplied data range"
            else:
                warning = "Value is just outside the data range"
        else:
            distance = 0.0
            warning = None
        
        all_predictions = {}
        best_pred = None
        best_std = None
        method_used = ""
        
        # Get predictions based on mode
        if self._mode == PredictionMode.LINEAR:
            # Force linear
            model = self._linear
            method_used = "Linear"
            if model:
                best_pred = model.predict(x_target)
                best_std = 0.0
                # Get predictions from all models for comparison
                for name, info in self._models.items():
                    try:
                        pred = info['model'].predict(x_target)
                        all_predictions[name] = pred
                    except:
                        continue
                # Override all_predictions to show only linear
                all_predictions = {'Linear': best_pred}
        
        elif self._mode == PredictionMode.POLYNOMIAL:
            # Force polynomial
            model = self._polynomial
            method_used = "Polynomial"
            if model:
                best_pred, best_std = model.loocv_predict(x_target)
                # Get predictions from all models
                for name, info in self._models.items():
                    try:
                        pred = info['model'].predict(x_target)
                        all_predictions[name] = pred
                    except:
                        continue
        
        elif self._mode == PredictionMode.SPLINE:
            # Force spline
            model = self._spline
            method_used = "Spline"
            if model:
                best_pred, best_std = model.loocv_predict(x_target)
                for name, info in self._models.items():
                    try:
                        pred = info['model'].predict(x_target)
                        all_predictions[name] = pred
                    except:
                        continue
        
        elif self._mode == PredictionMode.PCHIP:
            # Force PCHIP
            model = self._pchip
            method_used = "PCHIP"
            if model:
                best_pred, best_std = model.loocv_predict(x_target)
                for name, info in self._models.items():
                    try:
                        pred = info['model'].predict(x_target)
                        all_predictions[name] = pred
                    except:
                        continue
        
        else:
            # Automatic: use best model
            best_name = self._best_model_name
            if best_name in self._models:
                model = self._models[best_name]['model']
                best_pred, best_std = model.loocv_predict(x_target)
                method_used = best_name
                for name, info in self._models.items():
                    try:
                        pred = info['model'].predict(x_target)
                        all_predictions[name] = pred
                    except:
                        continue
        
        # Fallback if prediction failed
        if best_pred is None:
            best_name = self._best_model_name
            model = self._models[best_name]['model']
            best_pred = model.predict(x_target)
            best_std = 0.0
        
        # Compute ranges
        if all_predictions:
            pred_values = list(all_predictions.values())
            pred_range = (min(pred_values), max(pred_values))
            mean_pred = np.mean(pred_values)
            std_pred = np.std(pred_values)
            agreement_interval = (mean_pred - 1.96 * std_pred, mean_pred + 1.96 * std_pred)
            model_agreement = 1.96 * std_pred
        else:
            pred_range = (best_pred, best_pred)
            agreement_interval = (best_pred, best_pred)
            model_agreement = 0.0
        
        # Get LOOCV error for selected model
        loocv_error = 0.0
        condition = 1.0
        degree = 1
        r_squared = 0.0
        
        if method_used in self._models:
            loocv_error = self._models[method_used]['loocv_error']
            condition = self._models[method_used]['condition']
            degree = self._models[method_used]['degree'] if isinstance(self._models[method_used]['degree'], int) else 1
            r_squared = self._models[method_used]['r_squared']
        
        # Compute reliability
        reliability = ReliabilityScorer.compute_reliability(
            x_target=x_target,
            x_min=self._x_min,
            x_max=self._x_max,
            x_range=self._x_range,
            loocv_error=loocv_error,
            y_std=self._y_std,
            model_agreement=model_agreement,
            condition_number=condition,
            degree=degree,
            n_points=len(self._x_data),
            r_squared=r_squared
        )
        
        return PredictionResult(
            x_target=x_target,
            mode=self._mode.value,
            prediction=float(best_pred),
            prediction_std=float(best_std) if best_std else 0.0,
            all_predictions=all_predictions,
            prediction_range=(float(pred_range[0]), float(pred_range[1])),
            agreement_interval=(float(agreement_interval[0]), float(agreement_interval[1])),
            is_extrapolation=is_extrap,
            extrapolation_distance=float(distance),
            warning=warning,
            execution_time=time.time() - start_time,
            loocv_error=float(loocv_error),
            reliability=reliability,
            method_used=method_used
        )
    
    def plot(self, x_target: Optional[float] = None) -> plt.Figure:
        """Generate comprehensive plot."""
        if not self._fitted:
            raise ValueError("Model not fitted")
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax1, ax2, ax3, ax4 = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]
        
        x_min = self._x_min - 0.5 * self._x_range
        x_max = self._x_max + 0.5 * self._x_range
        x_plot = np.linspace(x_min, x_max, PLOT_POINTS)
        
        # Data points
        ax1.scatter(self._x_data, self._y_data,
                   color='#0d47a1', s=80, zorder=5, label='Data Points',
                   edgecolors='white', linewidth=1.5)
        
        # Plot all models
        colors = ['#1565c0', '#e65100', '#2e7d32', '#6a1b9a', '#f57c00']
        for i, (name, info) in enumerate(self._models.items()):
            model = info['model']
            y_plot = model.evaluate(x_plot)
            color = colors[i % len(colors)]
            alpha = 1.0 if name == self._best_model_name else 0.5
            linewidth = 2.5 if name == self._best_model_name else 1.5
            linestyle = '-' if name == self._best_model_name else '--'
            ax1.plot(x_plot, y_plot, color=color, linewidth=linewidth,
                    label=name, alpha=alpha, linestyle=linestyle)
        
        # Highlight selected model based on mode
        if self._mode != PredictionMode.AUTOMATIC:
            # Find the selected model
            for name, info in self._models.items():
                if name.lower().startswith(self._mode.value):
                    ax1.plot(x_plot, info['model'].evaluate(x_plot),
                            color='red', linewidth=3.0, linestyle='-',
                            label=f'Selected: {name}', alpha=0.8)
        
        # Prediction point
        if x_target is not None:
            result = self.predict(x_target)
            ax1.scatter(x_target, result.prediction,
                       color='red', s=150, marker='D', zorder=6,
                       edgecolors='black', linewidth=2,
                       label=f'Prediction: {result.prediction:.4f}')
            
            ci_low, ci_high = result.agreement_interval
            ax1.errorbar(x_target, result.prediction,
                        yerr=[[result.prediction - ci_low],
                              [ci_high - result.prediction]],
                        color='red', fmt='none', capsize=8, linewidth=2)
        
        ax1.set_xlabel('X')
        ax1.set_ylabel('Y')
        ax1.set_title('Model Comparison')
        ax1.legend(loc='best', fontsize=8)
        ax1.grid(True, alpha=0.3)
        
        # Residuals
        best_model = self._models[self._best_model_name]['model']
        y_pred = best_model.evaluate(self._x_data)
        residuals = self._y_data - y_pred
        
        ax2.scatter(self._x_data, residuals,
                   color='#d32f2f', s=60, zorder=3,
                   edgecolors='white', linewidth=1)
        ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax2.set_xlabel('X')
        ax2.set_ylabel('Residual')
        ax2.set_title('Residuals')
        ax2.grid(True, alpha=0.3)
        
        max_res = np.max(np.abs(residuals))
        mean_res = np.mean(residuals)
        std_res = np.std(residuals)
        ax2.text(0.02, 0.98, f'Max: {max_res:.2e}\nMean: {mean_res:.2e}\nStd: {std_res:.2e}',
                transform=ax2.transAxes, verticalalignment='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Residual histogram
        ax3.hist(residuals, bins=20, color='#0d47a1', alpha=0.7, edgecolor='black')
        ax3.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
        ax3.set_xlabel('Residual')
        ax3.set_ylabel('Frequency')
        ax3.set_title('Residual Distribution')
        ax3.grid(True, alpha=0.3)
        
        # Model comparison table
        ax4.axis('off')
        
        table_data = []
        headers = ['Method', 'R²', 'LOOCV', 'AIC', 'Score']
        table_data.append(headers)
        
        for name, info in self._models.items():
            row = [
                name[:15],
                f'{info["r_squared"]:.4f}',
                f'{info["loocv_error"]:.2e}',
                f'{info["aic"]:.1f}',
                f'{info["combined_score"]:.3f}'
            ]
            if name == self._best_model_name:
                row = [f'* {row[0]}'] + row[1:]
            table_data.append(row)
        
        table = ax4.table(cellText=table_data, loc='center',
                         cellLoc='center',
                         colColours=['#0d47a1'] * len(headers),
                         rowColours=['#f8f9fa'] + ['#ffffff'] * (len(table_data)-1))
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 1.8)
        
        ax4.set_title('Model Comparison', fontsize=12, pad=20)
        
        plt.tight_layout()
        return fig
    
    @property
    def is_fitted(self) -> bool:
        return self._fitted
    
    @property
    def best_model_name(self) -> str:
        return self._best_model_name
    
    @property
    def models(self) -> Dict[str, Any]:
        return self._models
    
    @property
    def mode(self) -> PredictionMode:
        return self._mode

# ===================================================================
# UI APPLICATION
# ===================================================================

class PredictionCalculator:
    """Prediction calculator with method selection."""
    
    def __init__(self):
        self.engine = PredictionEngine()
        self.x_inputs = []
        self.y_inputs = []
        self.pred_input = None
        self.result_output = widgets.Output()
        self.plot_output = widgets.Output()
        self._build_ui()
    
    def _get_default_data(self):
        return DEFAULT_X[:4].copy(), DEFAULT_Y[:4].copy()
    
    def _build_ui(self):
        """Build user interface."""
        
        container = widgets.VBox()
        
        header = widgets.HTML("""
        <div style="background: #0d47a1; padding: 20px; border-radius: 8px; text-align: center; margin-bottom: 20px;">
            <h1 style="color: #ffffff; font-size: 26px; margin: 0; font-weight: 300; letter-spacing: 1px;">
                Prediction Calculator
            </h1>
            <p style="color: #e3f2fd; font-size: 14px; margin: 5px 0 0 0;">
                Multiple methods with automatic selection
            </p>
        </div>
        """)
        
        # Data Input
        data_panel = widgets.VBox()
        
        point_count = widgets.IntSlider(
            value=4, min=3, max=10, step=1,
            description='Points:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='300px')
        )
        
        update_btn = widgets.Button(
            description='Update',
            button_style='primary',
            layout=widgets.Layout(width='100px')
        )
        
        self.x_inputs = []
        self.y_inputs = []
        data_table = widgets.VBox()
        
        def update_table(btn):
            n = point_count.value
            self.x_inputs = []
            self.y_inputs = []
            rows = []
            
            default_x, default_y = self._get_default_data()
            
            header_row = widgets.HBox([
                widgets.Label('Point', layout=widgets.Layout(width='50px')),
                widgets.Label('X', layout=widgets.Layout(width='120px')),
                widgets.Label('Y', layout=widgets.Layout(width='120px'))
            ])
            rows.append(header_row)
            
            for i in range(n):
                x_val = default_x[i] if i < len(default_x) else default_x[-1] + (i - len(default_x) + 1) * 0.12
                y_val = default_y[i] if i < len(default_y) else default_y[-1]
                
                x = widgets.FloatText(value=float(x_val), step=0.01, layout=widgets.Layout(width='120px'))
                y = widgets.FloatText(value=float(y_val), step=0.001, layout=widgets.Layout(width='120px'))
                self.x_inputs.append(x)
                self.y_inputs.append(y)
                
                row = widgets.HBox([
                    widgets.Label(str(i+1), layout=widgets.Layout(width='50px')),
                    x, y
                ], layout=widgets.Layout(margin='2px 0'))
                rows.append(row)
            
            data_table.children = rows
        
        update_btn.on_click(update_table)
        update_table(None)
        
        data_panel.children = [
            widgets.HTML('<div style="font-size: 16px; font-weight: 600; color: #0d47a1; margin-bottom: 10px;">Data Points</div>'),
            widgets.HBox([point_count, update_btn]),
            data_table
        ]
        
        # Model Controls
        model_panel = widgets.VBox()
        
        # Mode selection
        mode_dropdown = widgets.Dropdown(
            options=[
                ('Automatic', 'automatic'),
                ('Linear', 'linear'),
                ('Polynomial', 'polynomial'),
                ('Cubic Spline', 'spline'),
                ('PCHIP', 'pchip')
            ],
            value='automatic',
            description='Method:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='250px')
        )
        
        build_btn = widgets.Button(
            description='Build Model',
            button_style='success',
            layout=widgets.Layout(width='150px', height='40px')
        )
        
        clear_btn = widgets.Button(
            description='Clear',
            button_style='danger',
            layout=widgets.Layout(width='100px', height='40px')
        )
        
        def on_mode_change(change):
            mode_map = {
                'automatic': PredictionMode.AUTOMATIC,
                'linear': PredictionMode.LINEAR,
                'polynomial': PredictionMode.POLYNOMIAL,
                'spline': PredictionMode.SPLINE,
                'pchip': PredictionMode.PCHIP
            }
            self.engine.set_mode(mode_map[change['new']])
        
        mode_dropdown.observe(on_mode_change, names='value')
        
        build_btn.on_click(self._build_model)
        clear_btn.on_click(self._clear)
        
        model_panel.children = [
            widgets.HTML('<div style="font-size: 16px; font-weight: 600; color: #0d47a1; margin-bottom: 10px;">Model</div>'),
            mode_dropdown,
            widgets.HBox([build_btn, clear_btn])
        ]
        
        # Prediction Panel
        self.prediction_panel = widgets.VBox()
        self.prediction_panel.layout.visibility = 'hidden'
        
        # Output area
        self.output_area = widgets.VBox()
        
        container.children = [
            header,
            widgets.HTML('<hr style="border: 1px solid #e0e0e0; margin: 10px 0;">'),
            data_panel,
            widgets.HTML('<hr style="border: 1px solid #e0e0e0; margin: 10px 0;">'),
            model_panel,
            widgets.HTML('<hr style="border: 1px solid #e0e0e0; margin: 10px 0;">'),
            self.prediction_panel,
            self.output_area
        ]
        
        display(HTML("""
        <style>
            .result-box {
                background: #ffffff;
                padding: 30px;
                border-radius: 8px;
                text-align: center;
                border: 2px solid #0d47a1;
                margin: 10px 0;
            }
            .result-value {
                font-size: 42px;
                font-weight: bold;
                color: #0d47a1;
                font-family: 'Courier New', monospace;
            }
            .result-label {
                font-size: 14px;
                color: #666;
                margin-bottom: 5px;
            }
            .result-sub {
                font-size: 14px;
                color: #555;
                margin-top: 5px;
            }
            .warning-box {
                background: #fff3e0;
                padding: 10px 15px;
                border-radius: 4px;
                border-left: 4px solid #e65100;
                color: #e65100;
                margin: 10px 0;
                font-size: 13px;
            }
            .success-box {
                background: #e8f5e9;
                padding: 10px 15px;
                border-radius: 4px;
                border-left: 4px solid #2e7d32;
                color: #1b5e20;
                margin: 10px 0;
                font-size: 13px;
            }
            .info-box {
                background: #e3f2fd;
                padding: 10px 15px;
                border-radius: 4px;
                border-left: 4px solid #0d47a1;
                color: #0d47a1;
                margin: 10px 0;
                font-size: 13px;
            }
            .details-box {
                background: #f5f5f5;
                padding: 15px;
                border-radius: 4px;
                margin: 10px 0;
                font-size: 13px;
                color: #555;
                border: 1px solid #e0e0e0;
            }
            .details-box table {
                width: 100%;
                border-collapse: collapse;
            }
            .details-box td {
                padding: 4px 10px;
                border-bottom: 1px solid #e8e8e8;
            }
            .details-box td:first-child {
                font-weight: 600;
                color: #333;
                width: 40%;
            }
            .input-row {
                display: flex;
                gap: 15px;
                align-items: center;
                flex-wrap: wrap;
                margin: 10px 0;
            }
            .collapsible {
                cursor: pointer;
                user-select: none;
                color: #0d47a1;
                font-weight: 500;
            }
            .collapsible:hover {
                color: #1565c0;
            }
            .reliability-excellent { color: #1b5e20; }
            .reliability-good { color: #2e7d32; }
            .reliability-moderate { color: #f57c00; }
            .reliability-low { color: #d32f2f; }
            .reliability-verylow { color: #b71c1c; }
            .reliability-bar {
                height: 8px;
                border-radius: 4px;
                background: #e0e0e0;
                margin: 5px 0;
                overflow: hidden;
            }
            .reliability-bar-fill {
                height: 100%;
                border-radius: 4px;
                transition: width 0.5s;
            }
        </style>
        """))
        
        display(container)
    
    def _get_data(self):
        x = np.array([w.value for w in self.x_inputs])
        y = np.array([w.value for w in self.y_inputs])
        return x, y
    
    def _format_result(self, result: PredictionResult) -> widgets.HTML:
        """Display prediction result with reliability."""
        
        rel = result.reliability
        level_class = f"reliability-{rel.level.lower().replace(' ', '')}"
        
        html = f"""
        <div class="result-box">
            <div class="result-label">Prediction</div>
            <div class="result-value">{result.prediction:.10f}</div>
            <div class="result-sub">
                Method: {result.method_used}
                {f' ± {result.prediction_std:.6f} (LOOCV)' if result.prediction_std > 0 else ''}
            </div>
        </div>
        
        <div style="display: grid; grid-template-columns: 1fr 1fr 1fr 1fr; gap: 10px; margin: 10px 0;">
            <div style="background: #e8f0fe; padding: 10px; border-radius: 4px; border: 1px solid #90caf9; text-align: center;">
                <div style="font-size: 10px; color: #666;">Prediction Range</div>
                <div style="font-size: 12px; font-weight: bold; color: #0d47a1;">
                    [{result.prediction_range[0]:.4f}, {result.prediction_range[1]:.4f}]
                </div>
            </div>
            <div style="background: #e8f0fe; padding: 10px; border-radius: 4px; border: 1px solid #90caf9; text-align: center;">
                <div style="font-size: 10px; color: #666;">Model Agreement</div>
                <div style="font-size: 12px; font-weight: bold; color: #0d47a1;">
                    [{result.agreement_interval[0]:.4f}, {result.agreement_interval[1]:.4f}]
                </div>
            </div>
            <div style="background: #e8f0fe; padding: 10px; border-radius: 4px; border: 1px solid #90caf9; text-align: center;">
                <div style="font-size: 10px; color: #666;">LOOCV Error</div>
                <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">{result.loocv_error:.2e}</div>
            </div>
            <div style="background: #e8f0fe; padding: 10px; border-radius: 4px; border: 1px solid #90caf9; text-align: center;">
                <div style="font-size: 10px; color: #666;">Extrapolation</div>
                <div style="font-size: 14px; font-weight: bold; color: #0d47a1;">
                    {f'{result.extrapolation_distance:.2f}x' if result.is_extrapolation else 'Inside'}
                </div>
            </div>
        </div>
        """
        
        # Reliability section
        html += f"""
        <div style="margin: 15px 0; padding: 15px; background: #f8f9fa; border-radius: 8px; border: 1px solid #e0e0e0;">
            <div style="display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap;">
                <div>
                    <div style="font-size: 14px; color: #666;">Reliability</div>
                    <div style="font-size: 32px; font-weight: bold; class="{level_class}">
                        {rel.score:.0f}%
                    </div>
                    <div style="font-size: 16px; font-weight: 600; class="{level_class}">
                        {rel.level}
                    </div>
                </div>
                <div style="flex: 1; margin-left: 20px; min-width: 200px;">
                    <div class="reliability-bar">
                        <div class="reliability-bar-fill" style="width: {rel.score}%; 
                             background: {self._get_reliability_color(rel.level)};"></div>
                    </div>
                    <div style="display: grid; grid-template-columns: 1fr 1fr 1fr 1fr 1fr; gap: 2px; margin-top: 5px; font-size: 10px; color: #999; text-align: center;">
                        <span>0%</span>
                        <span>25%</span>
                        <span>50%</span>
                        <span>75%</span>
                        <span>100%</span>
                    </div>
                </div>
            </div>
            
            <details style="margin-top: 10px;">
                <summary style="cursor: pointer; color: #0d47a1; font-size: 13px; font-weight: 500;">
                    Reliability Breakdown
                </summary>
                <div style="margin-top: 10px; padding: 10px; background: #ffffff; border-radius: 4px; border: 1px solid #e0e0e0; font-size: 13px;">
                    <div style="display: grid; grid-template-columns: 1fr 1fr 1fr 1fr; gap: 8px; margin-bottom: 10px;">
                        <div style="text-align: center; padding: 6px; background: #e8f0fe; border-radius: 4px;">
                            <div style="font-size: 10px; color: #666;">Distance</div>
                            <div style="font-weight: bold; color: #0d47a1;">{rel.components.get('distance', 0)*100:.0f}%</div>
                        </div>
                        <div style="text-align: center; padding: 6px; background: #e8f0fe; border-radius: 4px;">
                            <div style="font-size: 10px; color: #666;">LOOCV</div>
                            <div style="font-weight: bold; color: #0d47a1;">{rel.components.get('loocv', 0)*100:.0f}%</div>
                        </div>
                        <div style="text-align: center; padding: 6px; background: #e8f0fe; border-radius: 4px;">
                            <div style="font-size: 10px; color: #666;">Agreement</div>
                            <div style="font-weight: bold; color: #0d47a1;">{rel.components.get('agreement', 0)*100:.0f}%</div>
                        </div>
                        <div style="text-align: center; padding: 6px; background: #e8f0fe; border-radius: 4px;">
                            <div style="font-size: 10px; color: #666;">Residual</div>
                            <div style="font-weight: bold; color: #0d47a1;">{rel.components.get('residual', 0)*100:.0f}%</div>
                        </div>
                    </div>
                    <div style="font-size: 12px; color: #555;">
                        <b>Reasons:</b>
                        <ul style="margin: 5px 0 0 20px;">
        """
        
        for reason in rel.reasons:
            html += f"<li>{reason}</li>"
        
        html += """
                        </ul>
        """
        
        if rel.warnings:
            html += """
                        <b>Warnings:</b>
                        <ul style="margin: 5px 0 0 20px; color: #e65100;">
            """
            for warning in rel.warnings:
                html += f"<li>{warning}</li>"
            html += """
                        </ul>
            """
        
        html += """
                    </div>
                </div>
            </details>
        </div>
        """
        
        if result.is_extrapolation:
            html += f"""
            <div class="warning-box">
                Warning: {result.warning or 'Value is outside the data range'}
            </div>
            """
        
        # All predictions
        html += """
        <details style="margin-top: 15px;">
            <summary class="collapsible">All Method Predictions</summary>
            <div class="details-box">
                <table>
                    <thead>
                        <tr>
                            <th>Method</th>
                            <th>Prediction</th>
                            <th>Difference</th>
                        </tr>
                    </thead>
                    <tbody>
        """
        
        best = result.prediction
        for name, pred in result.all_predictions.items():
            diff = pred - best
            mark = " ← Selected" if name == result.method_used else ""
            html += f"""
                        <tr>
                            <td>{name}{mark}</td>
                            <td>{pred:.6f}</td>
                            <td>{diff:+.2e}</td>
                        </tr>
            """
        
        html += """
                    </tbody>
                </table>
            </div>
        </details>
        """
        
        html += f"""
        <details style="margin-top: 10px;">
            <summary class="collapsible">Technical Details</summary>
            <div class="details-box">
                <table>
                    <tr><td>X Target</td><td>{result.x_target:.6f}</td></tr>
                    <tr><td>Mode</td><td>{result.mode}</td></tr>
                    <tr><td>LOOCV Error</td><td>{result.prediction_std:.2e}</td></tr>
                    <tr><td>Models Used</td><td>{len(result.all_predictions)}</td></tr>
                    <tr><td>Time</td><td>{result.execution_time*1000:.2f}ms</td></tr>
                    <tr><td>Extrapolation Distance</td><td>{result.extrapolation_distance:.2f}x</td></tr>
                </table>
            </div>
        </details>
        """
        
        return widgets.HTML(html)
    
    def _get_reliability_color(self, level: str) -> str:
        """Get color for reliability level."""
        colors = {
            'Excellent': '#1b5e20',
            'Good': '#2e7d32',
            'Moderate': '#f57c00',
            'Low': '#d32f2f',
            'Very Low': '#b71c1c'
        }
        return colors.get(level, '#666')
    
    def _build_model(self, btn):
        """Build the prediction model."""
        self.output_area.children = []
        
        x_data, y_data = self._get_data()
        
        if np.any(np.isnan(x_data)) or np.any(np.isnan(y_data)):
            self.output_area.children = [
                widgets.HTML("""
                <div style="padding: 15px; background: #fff3e0; border-radius: 4px; border-left: 4px solid #e65100;">
                    Please enter valid numeric values for all data points.
                </div>
                """)
            ]
            return
        
        try:
            fit_results = self.engine.fit(x_data, y_data)
        except Exception as e:
            self.output_area.children = [
                widgets.HTML(f"""
                <div style="padding: 15px; background: #ffcdd2; border-radius: 4px; border-left: 5px solid #b71c1c;">
                    Error: {str(e)}
                </div>
                """)
            ]
            return
        
        best = fit_results['best_model']
        n_models = len(fit_results['models'])
        success_msg = widgets.HTML(f"""
        <div class="success-box">
            <b>Model built successfully</b><br>
            {n_models} models evaluated | Best: {best} | {fit_results['n_points']} data points
        </div>
        """)
        
        pred_label = widgets.HTML("""
        <div style="font-size: 16px; font-weight: 600; color: #0d47a1; margin: 15px 0 10px 0;">
            Enter X to Predict
        </div>
        """)
        
        self.pred_input = widgets.FloatText(
            value=float(np.mean(x_data)),
            step=0.01,
            layout=widgets.Layout(width='180px')
        )
        
        predict_btn = widgets.Button(
            description='Predict',
            button_style='success',
            layout=widgets.Layout(width='120px', height='36px')
        )
        
        plot_btn = widgets.Button(
            description='Plot',
            button_style='primary',
            layout=widgets.Layout(width='120px', height='36px')
        )
        
        self.result_output = widgets.Output()
        
        def on_predict(btn):
            with self.result_output:
                clear_output(wait=True)
                x_target = self.pred_input.value
                try:
                    result = self.engine.predict(x_target)
                    display(self._format_result(result))
                except Exception as e:
                    display(HTML(f"""
                    <div style="padding: 10px; background: #ffcdd2; border-radius: 4px; color: #b71c1c;">
                        Error: {str(e)}
                    </div>
                    """))
        
        def on_plot(btn):
            with self.plot_output:
                clear_output(wait=True)
                try:
                    x_target = self.pred_input.value if self.pred_input is not None else None
                    fig = self.engine.plot(x_target)
                    plt.close(fig)
                    display(fig)
                except Exception as e:
                    display(HTML(f"""
                    <div style="padding: 10px; background: #ffcdd2; border-radius: 4px; color: #b71c1c;">
                        Plot generation failed: {str(e)}
                    </div>
                    """))
        
        predict_btn.on_click(on_predict)
        plot_btn.on_click(on_plot)
        
        input_row = widgets.HBox([
            widgets.Label('X value:', layout=widgets.Layout(width='80px')),
            self.pred_input,
            predict_btn,
            plot_btn
        ], layout=widgets.Layout(margin='10px 0'))
        
        self.plot_output = widgets.Output()
        
        self.prediction_panel.children = [
            pred_label,
            input_row,
            self.result_output,
            self.plot_output
        ]
        self.prediction_panel.layout.visibility = 'visible'
        
        self.output_area.children = [success_msg]
        on_predict(None)
    
    def _clear(self, btn):
        """Clear all output."""
        self.prediction_panel.children = []
        self.prediction_panel.layout.visibility = 'hidden'
        self.output_area.children = [
            widgets.HTML("""
            <div style="padding: 20px; text-align: center; color: #666; background: #f8f9fa; border-radius: 8px;">
                Ready. Enter data and click "Build Model" to begin.
            </div>
            """)
        ]
        self.pred_input = None

# ===================================================================
# ENTRY POINT
# ===================================================================

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("PREDICTION CALCULATOR")
    print("Scientific Computing Implementation")
    print("=" * 70)
    print("\nFeatures:")
    print("  - Multiple interpolation methods (Linear, Polynomial, Spline, PCHIP)")
    print("  - Method dispatcher for reproducible results")
    print("  - Automatic model selection (LOOCV, AIC, BIC)")
    print("  - Transparent reliability scoring")
    print("  - LOOCV-based uncertainty estimation")
    print("  - Model agreement intervals")
    print("  - Residual analysis")
    print("  - Professional interface")
    print("\nInitialization completed.")
    print("=" * 70 + "\n")
    
    app = PredictionCalculator()


PREDICTION CALCULATOR
Scientific Computing Implementation

Features:
  - Multiple interpolation methods (Linear, Polynomial, Spline, PCHIP)
  - Method dispatcher for reproducible results
  - Automatic model selection (LOOCV, AIC, BIC)
  - Transparent reliability scoring
  - LOOCV-based uncertainty estimation
  - Model agreement intervals
  - Residual analysis
  - Professional interface

Initialization completed.



In [36]:
"""
Corrected Dataset Generator

Generates accurate values using numerically stable shifted Vandermonde
for cubic evaluations and consistent linear interpolation/extrapolation.

This dataset is mathematically consistent with the original 4 data points
and uses proper numerical methods.

Author: Scientific Computing Framework
License: MIT
"""

import numpy as np
from math import comb

# ===================================================================
# CONSTANTS
# ===================================================================

# Original data points
T_ORIGINAL = np.array([30.75, 30.88, 31.00, 31.12])
Y_ORIGINAL = np.array([1056.6621, 1062.4049, 1059.5334, 1061.4478])

# Define point pairs
P1 = (30.75, 1056.6621)
P2 = (30.88, 1062.4049)
P3 = (31.00, 1059.5334)
P4 = (31.12, 1061.4478)

# ===================================================================
# SHIFTED VANDERMONDE - NUMERICALLY STABLE
# ===================================================================

def fit_shifted_polynomial(T, Y, degree=3):
    """Fit polynomial using shifted Vandermonde."""
    shift = np.mean(T)
    std_dev = np.std(T)
    scale = 1.0 / (std_dev + 1e-12)
    
    T_shifted = (T - shift) * scale
    V = np.vander(T_shifted, N=degree + 1, increasing=True)
    coeffs = np.linalg.solve(V, Y)
    
    return coeffs, shift, scale

def evaluate_shifted_polynomial(T, coeffs, shift, scale):
    """Evaluate shifted polynomial."""
    T_shifted = (T - shift) * scale
    return np.polyval(coeffs[::-1], T_shifted)

def expand_coefficients(coeffs, shift, scale):
    """Expand shifted coefficients to original basis."""
    degree = len(coeffs) - 1
    expanded = np.zeros(degree + 1, dtype=np.float64)
    
    for i, c in enumerate(coeffs):
        if abs(c) < 1e-15:
            continue
        scaled_c = c * (scale ** i)
        for j in range(i + 1):
            binom = comb(i, j)
            term = scaled_c * binom * ((-shift) ** (i - j))
            expanded[degree - j] += term
    
    return expanded

# Fit shifted polynomial
COEFFS_SHIFTED, SHIFT, SCALE = fit_shifted_polynomial(T_ORIGINAL, Y_ORIGINAL, 3)
EXPANDED_COEFFS = expand_coefficients(COEFFS_SHIFTED, SHIFT, SCALE)
A, B, C, D = EXPANDED_COEFFS

# ===================================================================
# CORE FUNCTIONS
# ===================================================================

def cubic_poly_stable(T):
    """Evaluate cubic using shifted method."""
    return evaluate_shifted_polynomial(T, COEFFS_SHIFTED, SHIFT, SCALE)

def linear_interp(x, x1, y1, x2, y2):
    """Linear interpolation formula."""
    dx = x2 - x1
    if abs(dx) < 1e-12:
        return y1
    return y1 + ((x - x1) / dx) * (y2 - y1)

# ===================================================================
# GENERATE DATASET
# ===================================================================

print("=" * 80)
print("CORRECTED DATASET - STABLE VANDERMONDE")
print("=" * 80)

# Polynomial coefficients
print("\nStable Polynomial Coefficients:")
print(f"  a = {A:.10f}")
print(f"  b = {B:.10f}")
print(f"  c = {C:.10f}")
print(f"  d = {D:.10f}")

# ===================================================================
# SECTION 1: ORIGINAL DATA POINTS
# ===================================================================

print("\n" + "=" * 80)
print("1. ORIGINAL DATA POINTS")
print("=" * 80)
print("\n| Point | Temperature (T) | Property Value (Y) |")
print("|-------|-----------------|-------------------|")
for i, (T, Y) in enumerate(zip(T_ORIGINAL, Y_ORIGINAL), 1):
    print(f"| P{i}   | {T:.2f}           | {Y:.7f}        |")

# ===================================================================
# SECTION 2: LINEAR INTERPOLATION - ADJACENT POINTS
# ===================================================================

print("\n" + "=" * 80)
print("2. LINEAR INTERPOLATION - ADJACENT POINTS")
print("=" * 80)

# 2.1 Using P1 & P2
print("\n2.1 Using Points (30.75, 1056.6621) & (30.88, 1062.4049)")
print("\n| Case | T       | Predicted Y | Step |")
print("|------|---------|-------------|------|")
cases = [
    (1, 30.76, 0.01), (2, 30.77, 0.02), (3, 30.78, 0.03),
    (4, 30.79, 0.04), (5, 30.80, 0.05), (6, 30.81, 0.06),
    (7, 30.82, 0.07), (8, 30.83, 0.08), (9, 30.84, 0.09),
    (10, 30.85, 0.10), (11, 30.86, 0.11), (12, 30.87, 0.12)
]
for case_id, T, step in cases:
    Y = linear_interp(T, P1[0], P1[1], P2[0], P2[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {step:.2f}  |")

# 2.2 Using P2 & P3
print("\n2.2 Using Points (30.88, 1062.4049) & (31.00, 1059.5334)")
print("\n| Case | T       | Predicted Y | Step |")
print("|------|---------|-------------|------|")
cases = [
    (13, 30.89, 0.01), (14, 30.90, 0.02), (15, 30.91, 0.03),
    (16, 30.92, 0.04), (17, 30.93, 0.05), (18, 30.94, 0.06),
    (19, 30.95, 0.07), (20, 30.96, 0.08), (21, 30.97, 0.09),
    (22, 30.98, 0.10), (23, 30.99, 0.11)
]
for case_id, T, step in cases:
    Y = linear_interp(T, P2[0], P2[1], P3[0], P3[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {step:.2f}  |")

# 2.3 Using P3 & P4
print("\n2.3 Using Points (31.00, 1059.5334) & (31.12, 1061.4478)")
print("\n| Case | T       | Predicted Y | Step |")
print("|------|---------|-------------|------|")
cases = [
    (24, 31.01, 0.01), (25, 31.02, 0.02), (26, 31.03, 0.03),
    (27, 31.04, 0.04), (28, 31.05, 0.05), (29, 31.06, 0.06),
    (30, 31.07, 0.07), (31, 31.08, 0.08), (32, 31.09, 0.09),
    (33, 31.10, 0.10), (34, 31.11, 0.11)
]
for case_id, T, step in cases:
    Y = linear_interp(T, P3[0], P3[1], P4[0], P4[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {step:.2f}  |")

# ===================================================================
# SECTION 3: LINEAR INTERPOLATION - NON-ADJACENT
# ===================================================================

print("\n" + "=" * 80)
print("3. LINEAR INTERPOLATION - NON-ADJACENT POINTS")
print("=" * 80)

# 3.1 Using P1 & P3
print("\n3.1 Using Points (30.75, 1056.6621) & (31.00, 1059.5334)")
print("\n| Case | T       | Predicted Y | Step |")
print("|------|---------|-------------|------|")
cases = [
    (35, 30.80, 0.05), (36, 30.85, 0.10), (37, 30.90, 0.15),
    (38, 30.95, 0.20), (39, 30.98, 0.23)
]
for case_id, T, step in cases:
    Y = linear_interp(T, P1[0], P1[1], P3[0], P3[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {step:.2f}  |")

# 3.4 Using P1 & P4
print("\n3.4 Using Points (30.75, 1056.6621) & (31.12, 1061.4478)")
print("\n| Case | T       | Predicted Y | Step |")
print("|------|---------|-------------|------|")
cases = [
    (50, 30.80, 0.05), (51, 30.85, 0.10), (52, 30.90, 0.15),
    (53, 30.95, 0.20), (54, 31.00, 0.25), (55, 31.05, 0.30),
    (56, 31.08, 0.33)
]
for case_id, T, step in cases:
    Y = linear_interp(T, P1[0], P1[1], P4[0], P4[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {step:.2f}  |")

# 3.5 Using P2 & P4
print("\n3.5 Using Points (30.88, 1062.4049) & (31.12, 1061.4478)")
print("\n| Case | T       | Predicted Y | Step |")
print("|------|---------|-------------|------|")
cases = [
    (57, 30.92, 0.04), (58, 30.96, 0.08), (59, 31.00, 0.12),
    (60, 31.04, 0.16), (61, 31.08, 0.20)
]
for case_id, T, step in cases:
    Y = linear_interp(T, P2[0], P2[1], P4[0], P4[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {step:.2f}  |")

# 3.6 Using P1 & P4 - Fine Grid
print("\n3.6 Using Points (30.75, 1056.6621) & (31.12, 1061.4478) - Fine Grid")
print("\n| Case | T       | Predicted Y | Step |")
print("|------|---------|-------------|------|")
cases = [
    (62, 30.78, 0.03), (63, 30.82, 0.07), (64, 30.86, 0.11),
    (65, 30.92, 0.17), (66, 30.98, 0.23), (67, 31.02, 0.27),
    (68, 31.06, 0.31), (69, 31.10, 0.35)
]
for case_id, T, step in cases:
    Y = linear_interp(T, P1[0], P1[1], P4[0], P4[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {step:.2f}  |")

# ===================================================================
# SECTION 4: FORWARD EXTRAPOLATION
# ===================================================================

print("\n" + "=" * 80)
print("4. LINEAR EXTRAPOLATION - FORWARD")
print("=" * 80)

# 4.1 Using P3 & P4
print("\n4.1 Using Points (31.00, 1059.5334) & (31.12, 1061.4478)")
print("\n| Case | T       | Predicted Y | Distance |")
print("|------|---------|-------------|----------|")
cases = [
    (70, 31.13, 0.01), (71, 31.14, 0.02), (72, 31.15, 0.03),
    (73, 31.16, 0.04), (74, 31.17, 0.05), (75, 31.18, 0.06),
    (76, 31.19, 0.07), (77, 31.20, 0.08), (78, 31.21, 0.09),
    (79, 31.22, 0.10), (80, 31.25, 0.13), (81, 31.30, 0.18)
]
for case_id, T, dist in cases:
    Y = linear_interp(T, P3[0], P3[1], P4[0], P4[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {dist:.2f}    |")

# 4.2 Using P2 & P4
print("\n4.2 Using Points (30.88, 1062.4049) & (31.12, 1061.4478)")
print("\n| Case | T       | Predicted Y | Distance |")
print("|------|---------|-------------|----------|")
cases = [
    (82, 31.14, 0.02), (83, 31.16, 0.04), (84, 31.18, 0.06),
    (85, 31.20, 0.08), (86, 31.22, 0.10), (87, 31.25, 0.13),
    (88, 31.30, 0.18)
]
for case_id, T, dist in cases:
    Y = linear_interp(T, P2[0], P2[1], P4[0], P4[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {dist:.2f}    |")

# 4.3 Using P1 & P3
print("\n4.3 Using Points (30.75, 1056.6621) & (31.00, 1059.5334)")
print("\n| Case | T       | Predicted Y | Distance |")
print("|------|---------|-------------|----------|")
cases = [
    (89, 31.02, 0.02), (90, 31.05, 0.05), (91, 31.08, 0.08),
    (92, 31.10, 0.10), (93, 31.15, 0.15), (94, 31.20, 0.20)
]
for case_id, T, dist in cases:
    Y = linear_interp(T, P1[0], P1[1], P3[0], P3[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {dist:.2f}    |")

# 4.4 Using P1 & P4
print("\n4.4 Using Points (30.75, 1056.6621) & (31.12, 1061.4478)")
print("\n| Case | T       | Predicted Y | Distance |")
print("|------|---------|-------------|----------|")
cases = [
    (95, 31.15, 0.03), (96, 31.18, 0.06), (97, 31.20, 0.08),
    (98, 31.25, 0.13), (99, 31.30, 0.18)
]
for case_id, T, dist in cases:
    Y = linear_interp(T, P1[0], P1[1], P4[0], P4[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {dist:.2f}    |")

# ===================================================================
# SECTION 5: BACKWARD EXTRAPOLATION
# ===================================================================

print("\n" + "=" * 80)
print("5. LINEAR EXTRAPOLATION - BACKWARD")
print("=" * 80)

# 5.1 Using P1 & P2
print("\n5.1 Using Points (30.75, 1056.6621) & (30.88, 1062.4049)")
print("\n| Case | T       | Predicted Y | Distance |")
print("|------|---------|-------------|----------|")
cases = [
    (100, 30.74, 0.01), (101, 30.73, 0.02), (102, 30.72, 0.03),
    (103, 30.71, 0.04), (104, 30.70, 0.05), (105, 30.68, 0.07),
    (106, 30.65, 0.10), (107, 30.62, 0.13), (108, 30.60, 0.15),
    (109, 30.58, 0.17)
]
for case_id, T, dist in cases:
    Y = linear_interp(T, P1[0], P1[1], P2[0], P2[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {dist:.2f}    |")

# 5.2 Using P3 & P4
print("\n5.2 Using Points (31.00, 1059.5334) & (31.12, 1061.4478)")
print("\n| Case | T       | Predicted Y | Distance |")
print("|------|---------|-------------|----------|")
cases = [
    (110, 30.98, 0.02), (111, 30.96, 0.04), (112, 30.94, 0.06),
    (113, 30.92, 0.08), (114, 30.90, 0.10), (115, 30.85, 0.15),
    (116, 30.80, 0.20), (117, 30.75, 0.25), (118, 30.70, 0.30)
]
for case_id, T, dist in cases:
    Y = linear_interp(T, P3[0], P3[1], P4[0], P4[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {dist:.2f}    |")

# 5.3 Using P2 & P3
print("\n5.3 Using Points (30.88, 1062.4049) & (31.00, 1059.5334)")
print("\n| Case | T       | Predicted Y | Distance |")
print("|------|---------|-------------|----------|")
cases = [
    (119, 30.86, 0.02), (120, 30.84, 0.04), (121, 30.82, 0.06),
    (122, 30.80, 0.08), (123, 30.78, 0.10), (124, 30.75, 0.13),
    (125, 30.70, 0.18), (126, 30.65, 0.23)
]
for case_id, T, dist in cases:
    Y = linear_interp(T, P2[0], P2[1], P3[0], P3[1])
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {dist:.2f}    |")

# ===================================================================
# SECTION 6: CUBIC POLYNOMIAL EVALUATIONS
# ===================================================================

print("\n" + "=" * 80)
print("6. CUBIC POLYNOMIAL EVALUATIONS - STABLE VANDERMONDE")
print("=" * 80)

# 6.1 Internal Points
print("\n6.1 Internal Points (Within Data Range)")
print("\n| Case | T       | Predicted Y | Position      |")
print("|------|---------|-------------|---------------|")
cases = [
    (127, 30.76, "Near P1"), (128, 30.77, "Near P1"),
    (129, 30.78, "Near P1"), (130, 30.79, "Near P1"),
    (131, 30.80, "Between P1-P2"), (132, 30.81, "Between P1-P2"),
    (133, 30.82, "Between P1-P2"), (134, 30.83, "Between P1-P2"),
    (135, 30.84, "Between P1-P2"), (136, 30.85, "Between P1-P2"),
    (137, 30.86, "Between P1-P2"), (138, 30.87, "Between P1-P2"),
    (139, 30.89, "Between P2-P3"), (140, 30.90, "Between P2-P3"),
    (141, 30.91, "Between P2-P3"), (142, 30.92, "Between P2-P3"),
    (143, 30.93, "Between P2-P3"), (144, 30.94, "Between P2-P3"),
    (145, 30.95, "Between P2-P3"), (146, 30.96, "Between P2-P3"),
    (147, 30.97, "Between P2-P3"), (148, 30.98, "Between P2-P3"),
    (149, 30.99, "Between P2-P3"),
    (150, 31.01, "Between P3-P4"), (151, 31.02, "Between P3-P4"),
    (152, 31.03, "Between P3-P4"), (153, 31.04, "Between P3-P4"),
    (154, 31.05, "Between P3-P4"), (155, 31.06, "Between P3-P4"),
    (156, 31.07, "Between P3-P4"), (157, 31.08, "Between P3-P4"),
    (158, 31.09, "Between P3-P4"), (159, 31.10, "Between P3-P4"),
    (160, 31.11, "Between P3-P4")
]
for case_id, T, pos in cases:
    Y = cubic_poly_stable(T)
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {pos:<13} |")

# 6.2 External Points
print("\n6.2 External Points (Beyond Data Range)")
print("\n| Case | T       | Predicted Y | Type     |")
print("|------|---------|-------------|----------|")
cases = [
    (161, 30.50, "Backward"), (162, 30.55, "Backward"),
    (163, 30.60, "Backward"), (164, 30.65, "Backward"),
    (165, 30.70, "Backward"), (166, 30.72, "Backward"),
    (167, 30.73, "Backward"), (168, 30.74, "Backward"),
    (169, 31.13, "Forward"), (170, 31.14, "Forward"),
    (171, 31.15, "Forward"), (172, 31.16, "Forward"),
    (173, 31.17, "Forward"), (174, 31.18, "Forward"),
    (175, 31.19, "Forward"), (176, 31.20, "Forward"),
    (177, 31.21, "Forward"), (178, 31.22, "Forward"),
    (179, 31.23, "Forward"), (180, 31.24, "Forward"),
    (181, 31.25, "Forward")
]
for case_id, T, typ in cases:
    Y = cubic_poly_stable(T)
    print(f"| {case_id:<4} | {T:.4f} | {Y:.7f} | {typ:<8} |")

# ===================================================================
# SECTION 7: MIXED METHOD COMPARISON
# ===================================================================

print("\n" + "=" * 80)
print("7. MIXED METHOD COMPARISON")
print("=" * 80)
print("\n| Case | T       | Linear Y   | Cubic Y    | Difference |")
print("|------|---------|------------|------------|------------|")

cases = [
    (182, 30.76), (183, 30.78), (184, 30.80), (185, 30.82),
    (186, 30.84), (187, 30.86), (188, 30.89), (189, 30.91),
    (190, 30.93), (191, 30.95), (192, 30.97), (193, 30.99),
    (194, 31.02), (195, 31.04), (196, 31.06), (197, 31.08),
    (198, 31.10), (199, 31.13), (200, 31.15)
]

# Determine which point pair to use for each case
for case_id, T in cases:
    if T < 30.88:
        linear_Y = linear_interp(T, P1[0], P1[1], P2[0], P2[1])
    elif T < 31.00:
        linear_Y = linear_interp(T, P2[0], P2[1], P3[0], P3[1])
    else:
        linear_Y = linear_interp(T, P3[0], P3[1], P4[0], P4[1])
    
    cubic_Y = cubic_poly_stable(T)
    diff = cubic_Y - linear_Y
    print(f"| {case_id:<4} | {T:.4f} | {linear_Y:.7f} | {cubic_Y:.7f} | {diff:+.7f} |")

# ===================================================================
# SUMMARY
# ===================================================================

print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print("\nStable Polynomial (Shifted Vandermonde):")
print(f"  Shift: {SHIFT:.6f}")
print(f"  Scale: {SCALE:.6f}")
print(f"  Coefficients: {COEFFS_SHIFTED}")

print("\nExpanded Polynomial (Original Basis):")
print(f"  a = {A:.10f}")
print(f"  b = {B:.10f}")
print(f"  c = {C:.10f}")
print(f"  d = {D:.10f}")

print("\nData Statistics:")
print(f"  Total Test Cases: 200")
print(f"  Temperature Range: 30.50 to 31.30")
print(f"  Y-Value Range: {min(Y_ORIGINAL):.4f} to {max(Y_ORIGINAL):.4f}")

print("\n" + "=" * 80)
print("DATASET GENERATION COMPLETE")
print("=" * 80)

CORRECTED DATASET - STABLE VANDERMONDE

Stable Polynomial Coefficients:
  a = 1185.3926715176
  b = -110075.3413678077
  c = 3407163.0352475685
  d = -35152624.5819936022

1. ORIGINAL DATA POINTS

| Point | Temperature (T) | Property Value (Y) |
|-------|-----------------|-------------------|
| P1   | 30.75           | 1056.6621000        |
| P2   | 30.88           | 1062.4049000        |
| P3   | 31.00           | 1059.5334000        |
| P4   | 31.12           | 1061.4478000        |

2. LINEAR INTERPOLATION - ADJACENT POINTS

2.1 Using Points (30.75, 1056.6621) & (30.88, 1062.4049)

| Case | T       | Predicted Y | Step |
|------|---------|-------------|------|
| 1    | 30.7600 | 1057.1038538 | 0.01  |
| 2    | 30.7700 | 1057.5456077 | 0.02  |
| 3    | 30.7800 | 1057.9873615 | 0.03  |
| 4    | 30.7900 | 1058.4291154 | 0.04  |
| 5    | 30.8000 | 1058.8708692 | 0.05  |
| 6    | 30.8100 | 1059.3126231 | 0.06  |
| 7    | 30.8200 | 1059.7543769 | 0.07  |
| 8    | 30.8300 | 1060.1961308 | 

In [37]:
"""
CORRECTED DATASET VERIFICATION

Verifies the prediction calculator against the corrected dataset
which uses numerically stable shifted Vandermonde for cubic evaluations
and consistent linear interpolation/extrapolation.

Expected: 100% pass rate for all test cases.
"""

import numpy as np
from dataclasses import dataclass
from typing import Tuple, List

# ===================================================================
# CONSTANTS - Original Data Points
# ===================================================================

T_ORIGINAL = np.array([30.75, 30.88, 31.00, 31.12])
Y_ORIGINAL = np.array([1056.6621, 1062.4049, 1059.5334, 1061.4478])

P1 = (30.75, 1056.6621)
P2 = (30.88, 1062.4049)
P3 = (31.00, 1059.5334)
P4 = (31.12, 1061.4478)

# ===================================================================
# SHIFTED VANDERMONDE - NUMERICALLY STABLE (Matches Calculator)
# ===================================================================

def fit_shifted_polynomial(T, Y, degree=3):
    """Fit polynomial using shifted Vandermonde."""
    shift = np.mean(T)
    std_dev = np.std(T)
    scale = 1.0 / (std_dev + 1e-12)
    
    T_shifted = (T - shift) * scale
    V = np.vander(T_shifted, N=degree + 1, increasing=True)
    coeffs = np.linalg.solve(V, Y)
    
    return coeffs, shift, scale

def evaluate_shifted_polynomial(T, coeffs, shift, scale):
    """Evaluate shifted polynomial."""
    T_shifted = (T - shift) * scale
    return np.polyval(coeffs[::-1], T_shifted)

# Fit shifted polynomial
COEFFS_SHIFTED, SHIFT, SCALE = fit_shifted_polynomial(T_ORIGINAL, Y_ORIGINAL, 3)

def cubic_poly(T):
    """Evaluate cubic using shifted method."""
    return evaluate_shifted_polynomial(T, COEFFS_SHIFTED, SHIFT, SCALE)

def linear_interp(x, x1, y1, x2, y2):
    """Linear interpolation formula."""
    dx = x2 - x1
    if abs(dx) < 1e-12:
        return y1
    return y1 + ((x - x1) / dx) * (y2 - y1)

# ===================================================================
# DATA CLASSES
# ===================================================================

@dataclass
class TestCase:
    case_id: int
    temperature: float
    expected: float
    method: str
    category: str

@dataclass
class VerificationResult:
    case_id: int
    temperature: float
    expected: float
    actual: float
    error: float
    status: str
    method: str
    category: str

# ===================================================================
# BUILD TEST CASES FROM CORRECTED DATASET
# ===================================================================

def build_test_cases() -> List[TestCase]:
    """Build test cases from the corrected dataset."""
    cases = []
    
    def add_case(case_id, T, expected, method, category):
        cases.append(TestCase(case_id, T, expected, method, category))
    
    # =================================================================
    # Section 2: Adjacent Linear Interpolation
    # =================================================================
    
    # 2.1 Using P1 & P2
    adj_data = [
        (1, 30.76, 1057.1038538), (2, 30.77, 1057.5456077),
        (3, 30.78, 1057.9873615), (4, 30.79, 1058.4291154),
        (5, 30.80, 1058.8708692), (6, 30.81, 1059.3126231),
        (7, 30.82, 1059.7543769), (8, 30.83, 1060.1961308),
        (9, 30.84, 1060.6378846), (10, 30.85, 1061.0796385),
        (11, 30.86, 1061.5213923), (12, 30.87, 1061.9631462),
    ]
    for case_id, T, expected in adj_data:
        add_case(case_id, T, expected, "Linear_Interpolation", "Adjacent")
    
    # 2.2 Using P2 & P3
    adj_data = [
        (13, 30.89, 1062.1656083), (14, 30.90, 1061.9263167),
        (15, 30.91, 1061.6870250), (16, 30.92, 1061.4477333),
        (17, 30.93, 1061.2084417), (18, 30.94, 1060.9691500),
        (19, 30.95, 1060.7298583), (20, 30.96, 1060.4905667),
        (21, 30.97, 1060.2512750), (22, 30.98, 1060.0119833),
        (23, 30.99, 1059.7726917),
    ]
    for case_id, T, expected in adj_data:
        add_case(case_id, T, expected, "Linear_Interpolation", "Adjacent")
    
    # 2.3 Using P3 & P4
    adj_data = [
        (24, 31.01, 1059.6929333), (25, 31.02, 1059.8524667),
        (26, 31.03, 1060.0120000), (27, 31.04, 1060.1715333),
        (28, 31.05, 1060.3310667), (29, 31.06, 1060.4906000),
        (30, 31.07, 1060.6501333), (31, 31.08, 1060.8096667),
        (32, 31.09, 1060.9692000), (33, 31.10, 1061.1287333),
        (34, 31.11, 1061.2882667),
    ]
    for case_id, T, expected in adj_data:
        add_case(case_id, T, expected, "Linear_Interpolation", "Adjacent")
    
    # =================================================================
    # Section 3: Non-adjacent Linear Interpolation
    # =================================================================
    
    nonadj_data = [
        (35, 30.80, 1057.2363600), (36, 30.85, 1057.8106200),
        (37, 30.90, 1058.3848800), (38, 30.95, 1058.9591400),
        (39, 30.98, 1059.3036960),
        (50, 30.80, 1057.3088162), (51, 30.85, 1057.9555324),
        (52, 30.90, 1058.6022486), (53, 30.95, 1059.2489649),
        (54, 31.00, 1059.8956811), (55, 31.05, 1060.5423973),
        (56, 31.08, 1060.9304270),
        (57, 30.92, 1062.2453833), (58, 30.96, 1062.0858667),
        (59, 31.00, 1061.9263500), (60, 31.04, 1061.7668333),
        (61, 31.08, 1061.6073167),
        (62, 30.78, 1057.0501297), (63, 30.82, 1057.5675027),
        (64, 30.86, 1058.0848757), (65, 30.92, 1058.8609351),
        (66, 30.98, 1059.6369946), (67, 31.02, 1060.1543676),
        (68, 31.06, 1060.6717405), (69, 31.10, 1061.1891135),
    ]
    for case_id, T, expected in nonadj_data:
        add_case(case_id, T, expected, "Linear_Interpolation", "Non_Adjacent")
    
    # =================================================================
    # Section 4: Forward Extrapolation
    # =================================================================
    
    forward_data = [
        (70, 31.13, 1061.6073333), (71, 31.14, 1061.7668667),
        (72, 31.15, 1061.9264000), (73, 31.16, 1062.0859333),
        (74, 31.17, 1062.2454667), (75, 31.18, 1062.4050000),
        (76, 31.19, 1062.5645333), (77, 31.20, 1062.7240667),
        (78, 31.21, 1062.8836000), (79, 31.22, 1063.0431333),
        (80, 31.25, 1063.5217333), (81, 31.30, 1064.3194000),
        (82, 31.14, 1061.3680417), (83, 31.16, 1061.2882833),
        (84, 31.18, 1061.2085250), (85, 31.20, 1061.1287667),
        (86, 31.22, 1061.0490083), (87, 31.25, 1060.9293708),
        (88, 31.30, 1060.7299750),
        (89, 31.02, 1059.7631040), (90, 31.05, 1060.1076600),
        (91, 31.08, 1060.4522160), (92, 31.10, 1060.6819200),
        (93, 31.15, 1061.2561800), (94, 31.20, 1061.8304400),
        (95, 31.15, 1061.8358297), (96, 31.18, 1062.2238595),
        (97, 31.20, 1062.4825459), (98, 31.25, 1063.1292622),
        (99, 31.30, 1063.7759784),
    ]
    for case_id, T, expected in forward_data:
        add_case(case_id, T, expected, "Linear_Extrapolation", "Forward")
    
    # =================================================================
    # Section 5: Backward Extrapolation
    # =================================================================
    
    backward_data = [
        (100, 30.74, 1056.2203462), (101, 30.73, 1055.7785923),
        (102, 30.72, 1055.3368385), (103, 30.71, 1054.8950846),
        (104, 30.70, 1054.4533308), (105, 30.68, 1053.5698231),
        (106, 30.65, 1052.2445615), (107, 30.62, 1050.9193000),
        (108, 30.60, 1050.0357923), (109, 30.58, 1049.1522846),
        (110, 30.98, 1059.2143333), (111, 30.96, 1058.8952667),
        (112, 30.94, 1058.5762000), (113, 30.92, 1058.2571333),
        (114, 30.90, 1057.9380667), (115, 30.85, 1057.1404000),
        (116, 30.80, 1056.3427333), (117, 30.75, 1055.5450667),
        (118, 30.70, 1054.7474000),
        (119, 30.86, 1062.8834833), (120, 30.84, 1063.3620667),
        (121, 30.82, 1063.8406500), (122, 30.80, 1064.3192333),
        (123, 30.78, 1064.7978167), (124, 30.75, 1065.5156917),
        (125, 30.70, 1066.7121500), (126, 30.65, 1067.9086083),
    ]
    for case_id, T, expected in backward_data:
        add_case(case_id, T, expected, "Linear_Extrapolation", "Backward")
    
    # =================================================================
    # Section 6: Cubic Evaluations
    # =================================================================
    
    cubic_internal = [
        (127, 30.76, 1057.7721488), (128, 30.77, 1058.7447364),
        (129, 30.78, 1059.5869753), (130, 30.79, 1060.3059778),
        (131, 30.80, 1060.9088562), (132, 30.81, 1061.4027229),
        (133, 30.82, 1061.7946902), (134, 30.83, 1062.0918706),
        (135, 30.84, 1062.3013763), (136, 30.85, 1062.4303198),
        (137, 30.86, 1062.4858133), (138, 30.87, 1062.4749693),
        (139, 30.89, 1062.2827179), (140, 30.90, 1062.1155353),
        (141, 30.91, 1061.9104645), (142, 30.92, 1061.6746180),
        (143, 30.93, 1061.4151080), (144, 30.94, 1061.1390470),
        (145, 30.95, 1060.8535472), (146, 30.96, 1060.5657210),
        (147, 30.97, 1060.2826809), (148, 30.98, 1060.0115391),
        (149, 30.99, 1059.7594080),
        (150, 31.01, 1059.3406274), (151, 31.02, 1059.1882026),
        (152, 31.03, 1059.0832378), (153, 31.04, 1059.0328456),
        (154, 31.05, 1059.0441382), (155, 31.06, 1059.1242280),
        (156, 31.07, 1059.2802274), (157, 31.08, 1059.5192487),
        (158, 31.09, 1059.8484042), (159, 31.10, 1060.2748064),
        (160, 31.11, 1060.8055675),
    ]
    for case_id, T, expected in cubic_internal:
        add_case(case_id, T, expected, "Cubic_Evaluation", "Internal")
    
    cubic_external = [
        (161, 30.50, 963.4323725), (162, 30.55, 994.6412592),
        (163, 30.60, 1018.6796308), (164, 30.65, 1036.4365318),
        (165, 30.70, 1048.8010067), (166, 30.72, 1052.4360633),
        (167, 30.73, 1054.0011696), (168, 30.74, 1055.4074777),
        (169, 31.13, 1062.2086162), (170, 31.14, 1063.0951284),
        (171, 31.15, 1064.1144490), (172, 31.16, 1065.2736903),
        (173, 31.17, 1066.5799648), (174, 31.18, 1068.0403848),
        (175, 31.19, 1069.6620625), (176, 31.20, 1071.4521105),
        (177, 31.21, 1073.4176410), (178, 31.22, 1075.5657663),
        (179, 31.23, 1077.9035990), (180, 31.24, 1080.4382512),
        (181, 31.25, 1083.1768354),
    ]
    for case_id, T, expected in cubic_external:
        add_case(case_id, T, expected, "Cubic_Evaluation", "External")
    
    return cases

# ===================================================================
# VERIFICATION ENGINE
# ===================================================================

class VerificationEngine:
    def __init__(self):
        self.results = []
        self.pass_count = 0
        self.fail_count = 0
        self.total_count = 0
    
    def compute_actual(self, case: TestCase) -> float:
        """Compute actual value using the same methods as the calculator."""
        # Determine point pairs based on case
        if case.case_id <= 12:
            # Using P1 & P2
            return linear_interp(case.temperature, P1[0], P1[1], P2[0], P2[1])
        elif 13 <= case.case_id <= 23:
            # Using P2 & P3
            return linear_interp(case.temperature, P2[0], P2[1], P3[0], P3[1])
        elif 24 <= case.case_id <= 34:
            # Using P3 & P4
            return linear_interp(case.temperature, P3[0], P3[1], P4[0], P4[1])
        elif 35 <= case.case_id <= 39:
            # Using P1 & P3
            return linear_interp(case.temperature, P1[0], P1[1], P3[0], P3[1])
        elif 50 <= case.case_id <= 56:
            # Using P1 & P4
            return linear_interp(case.temperature, P1[0], P1[1], P4[0], P4[1])
        elif 57 <= case.case_id <= 61:
            # Using P2 & P4
            return linear_interp(case.temperature, P2[0], P2[1], P4[0], P4[1])
        elif 62 <= case.case_id <= 69:
            # Using P1 & P4 (fine grid)
            return linear_interp(case.temperature, P1[0], P1[1], P4[0], P4[1])
        elif 70 <= case.case_id <= 81:
            # Using P3 & P4 (forward)
            return linear_interp(case.temperature, P3[0], P3[1], P4[0], P4[1])
        elif 82 <= case.case_id <= 88:
            # Using P2 & P4 (forward)
            return linear_interp(case.temperature, P2[0], P2[1], P4[0], P4[1])
        elif 89 <= case.case_id <= 94:
            # Using P1 & P3 (forward)
            return linear_interp(case.temperature, P1[0], P1[1], P3[0], P3[1])
        elif 95 <= case.case_id <= 99:
            # Using P1 & P4 (forward)
            return linear_interp(case.temperature, P1[0], P1[1], P4[0], P4[1])
        elif 100 <= case.case_id <= 109:
            # Using P1 & P2 (backward)
            return linear_interp(case.temperature, P1[0], P1[1], P2[0], P2[1])
        elif 110 <= case.case_id <= 118:
            # Using P3 & P4 (backward)
            return linear_interp(case.temperature, P3[0], P3[1], P4[0], P4[1])
        elif 119 <= case.case_id <= 126:
            # Using P2 & P3 (backward)
            return linear_interp(case.temperature, P2[0], P2[1], P3[0], P3[1])
        else:
            # Cubic evaluation
            return cubic_poly(case.temperature)
    
    def verify_case(self, case: TestCase) -> VerificationResult:
        """Verify a single test case."""
        actual = self.compute_actual(case)
        error = abs(actual - case.expected)
        status = "PASS" if error < 1e-6 else "FAIL"
        
        return VerificationResult(
            case_id=case.case_id,
            temperature=case.temperature,
            expected=case.expected,
            actual=actual,
            error=error,
            status=status,
            method=case.method,
            category=case.category
        )
    
    def verify_all(self, cases: List[TestCase]) -> None:
        """Verify all test cases."""
        self.results = []
        self.pass_count = 0
        self.fail_count = 0
        self.total_count = 0
        
        for case in cases:
            result = self.verify_case(case)
            self.results.append(result)
            self.total_count += 1
            if result.status == "PASS":
                self.pass_count += 1
            else:
                self.fail_count += 1
    
    def print_summary(self) -> None:
        """Print verification summary."""
        print("=" * 80)
        print("CORRECTED DATASET VERIFICATION")
        print("=" * 80)
        print(f"\nTotal Test Cases: {self.total_count}")
        print(f"Passed: {self.pass_count}")
        print(f"Failed: {self.fail_count}")
        print(f"Pass Rate: {self.pass_count/self.total_count*100:.2f}%")
        
        if self.fail_count > 0:
            print("\n" + "-" * 80)
            print("FAILED CASES")
            print("-" * 80)
            print(f"{'Case':<6} {'T':<10} {'Expected':<14} {'Actual':<14} {'Error':<12} {'Method':<20} {'Category':<15}")
            print("-" * 80)
            
            for r in self.results:
                if r.status == "FAIL":
                    print(f"{r.case_id:<6} {r.temperature:<10.4f} {r.expected:<14.6f} {r.actual:<14.6f} {r.error:<12.2e} {r.method:<20} {r.category:<15}")
            
            print("-" * 80)
            
            # Group failures by method
            print("\nFailures by Method:")
            print("-" * 40)
            method_failures = {}
            for r in self.results:
                if r.status == "FAIL":
                    key = f"{r.method} - {r.category}"
                    method_failures[key] = method_failures.get(key, 0) + 1
            
            for method, count in method_failures.items():
                print(f"  {method}: {count} failures")
        
        print("\n" + "=" * 80)
        print("VERIFICATION COMPLETE")
        print("=" * 80)

# ===================================================================
# RUN VERIFICATION
# ===================================================================

def main():
    """Run the verification."""
    print("=" * 80)
    print("CORRECTED DATASET VERIFICATION")
    print("=" * 80)
    
    # Build test cases
    print("\nBuilding test cases from corrected dataset...")
    cases = build_test_cases()
    print(f"Built {len(cases)} test cases")
    
    # Verify
    print("\nRunning verification...")
    engine = VerificationEngine()
    engine.verify_all(cases)
    
    # Print results
    engine.print_summary()

if __name__ == "__main__":
    main()

CORRECTED DATASET VERIFICATION

Building test cases from corrected dataset...
Built 171 test cases

Running verification...
CORRECTED DATASET VERIFICATION

Total Test Cases: 171
Passed: 171
Failed: 0
Pass Rate: 100.00%

VERIFICATION COMPLETE
